# The Effect of Ridge on Multicollinearity
### The Task: Create or find a dataset with highly correlated features (multicollinearity).
## Objectives:
#### Train a standard Linear Regression model and a Ridge Regression model.
#### Compare the magnitude of the coefficients between the two models. Observe how Ridge penalizes and shrinks the coefficients of correlated features to stabilize the model.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split

# 1. Load the dataset with highly correlated features
df = pd.read_csv('Multicollinear.csv')
X = df[['Feature_1', 'Feature_2', 'Feature_3']]

# 2. Synthesize a target variable (y) since it's missing in the CSV
np.random.seed(42)
y = 10 * X['Feature_1'] + 5 * X['Feature_2'] + 2 * X['Feature_3'] + np.random.normal(0, 0.5, size=len(X))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train Standard Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# 4. Train Ridge Regression
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train, y_train)

print("--- Multicollinearity: Coefficient Comparison ---")
print(f"Linear Regression Coefficients: {lin_reg.coef_}")
print(f"Ridge Regression Coefficients:  {ridge_reg.coef_}")

--- Multicollinearity: Coefficient Comparison ---
Linear Regression Coefficients: [10.74710719  4.23612504  2.75628944]
Ridge Regression Coefficients:  [5.00131615 6.94770805 4.00283546]


Observation & Explanation
When features are highly correlated (multicollinearity), standard Linear Regression struggles to determine which feature is actually responsible for changes in the target. It often assigns wildly unstable or arbitrary coefficients (in this test run, it assigned 10.74, 4.23, and 2.75).

Ridge Regression, however, applies an L2 penalty, which mathematically punishes overly large coefficients. Because it wants to keep all coefficients as small as possible, it responds to the highly correlated data by shrinking and distributing the weights more evenly across the features (outputting 5.00, 6.94, and 4.00). This prevents any single feature from aggressively dominating the model and makes the entire system much more stable and reliable when predicting new data.

# Feature Selection with Lasso CV
### Dataset: from sklearn.datasets import make_regression
#### The Task: Use a dataset with many features, some of which are irrelevant (e.g., adding pure noise columns to a housing dataset).
## Objectives:
#### Use LassoCV to automatically find the best regularization strength ( / alpha) via cross-validation.
#### Examine the final coefficients. Identify which features Lasso pushed exactly to zero, effectively performing automatic feature selection.

In [3]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split

# 1. Create a dataset with 100 features, but only 10 are actually useful (informative)
X, y = make_regression(n_samples=500, n_features=100, n_informative=10, noise=0.1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Use LassoCV to automatically find the best regularization strength (alpha)
# cv=5 means it breaks the training data into 5 folds to test itself
lasso_cv = LassoCV(cv=5, random_state=42)
lasso_cv.fit(X_train, y_train)

# 3. Examine the final coefficients
zero_coefs = np.sum(lasso_cv.coef_ == 0)
non_zero_coefs = np.sum(lasso_cv.coef_ != 0)

print("--- LassoCV Feature Selection ---")
print(f"Optimal Alpha (Regularization Strength): {lasso_cv.alpha_:.4f}")
print(f"Features eliminated (pushed to 0): {zero_coefs} out of 100")
print(f"Features retained (non-zero): {non_zero_coefs} out of 100")

--- LassoCV Feature Selection ---
Optimal Alpha (Regularization Strength): 0.0994
Features eliminated (pushed to 0): 90 out of 100
Features retained (non-zero): 10 out of 100


Observation & Explanation
Running this script demonstrates the superpower of Lasso Regression (L1 penalty). Unlike Ridge, which only shrinks coefficients closer to zero, Lasso can push them exactly to zero.

During the run, LassoCV tested multiple regularization strengths and found the mathematically optimal alpha (roughly 0.099). Once it applied this penalty, it successfully identified and eliminated exactly 90 noise features by pushing their coefficients to 0.0. It kept only the 10 features that actually had predictive value. In plain English: Lasso acts as an automated feature selector, cleaning up your dataset for you and ignoring irrelevant data columns entirely!